
# Bounding-Box Crops

This notebook reads a VIA-style annotation CSV and produces **padded bounding-box crops** per label.

**What you provide:**
- `CSV_PATH`: path to your VIA CSV (e.g., `CBVD-5.csv`)
- `IMG_ROOT`: folder containing the frames referenced in the CSV (e.g., `data/labelframes/`)
- `OUT_ROOT`: destination for crops (e.g., `workdir/crops_raw/`)

**Output:**
- `OUT_ROOT/<label>/*.jpg` cropped images with a small padding.


In [1]:
# Core Python & data handling
import json, ast, os, time
from pathlib import Path
from collections import defaultdict

# Data processing
import pandas as pd
import cv2
import numpy as np

# Progress tracking
from tqdm import tqdm

In [2]:
# Configuration
CSV_PATH = Path("data/CBVD-5.csv")
IMG_ROOT = Path("data/labelframes/labelframes")
OUT_ROOT = Path("workdir/crops_raw")
PAD_FRACTION = 0.08
SKIP_ROWS = 9

# Behavior mapping
BEHAVIOR_CODES = {
    0: "stand", 1: "lying down", 2: "foraging", 
    3: "drinking water", 4: "rumination"
}
BEHAVIOR_PRIORITY = ["drinking water", "foraging", "rumination", "lying down", "stand"]

OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Processing {CSV_PATH} → {OUT_ROOT}")

Processing data/CBVD-5.csv → workdir/crops_raw


In [3]:
# Helper functions
def parse_file_list(s):
    if isinstance(s, list): return s[0] if s else None
    try: return ast.literal_eval(s)[0] if ast.literal_eval(s) else None
    except: return s

def parse_box(spatial_coordinates):
    coords = json.loads(spatial_coordinates) if isinstance(spatial_coordinates, str) else spatial_coordinates
    if isinstance(coords[0], list): coords = coords[0]
    _, x, y, w, h = coords
    return int(x), int(y), int(w), int(h)

def extract_behavior(metadata):
    try:
        meta_dict = json.loads(metadata) if isinstance(metadata, str) else ast.literal_eval(metadata)
        behavior_codes_str = meta_dict.get("1") or meta_dict.get(1)
        if not behavior_codes_str: return "unknown"
        
        codes = [int(c) for c in str(behavior_codes_str).split(",") if c.strip().isdigit()]
        behaviors = [BEHAVIOR_CODES[c] for c in codes if c in BEHAVIOR_CODES]
        
        for priority in BEHAVIOR_PRIORITY:
            if priority in behaviors: return priority
        return behaviors[0] if behaviors else "unknown"
    except: return "unknown"

def get_padded_crop(x, y, w, h, img_w, img_h, pad_frac):
    pad_w, pad_h = max(1, int(w * pad_frac)), max(1, int(h * pad_frac))
    x0, y0 = max(0, x - pad_w), max(0, y - pad_h)
    x1, y1 = min(img_w, x + w + pad_w), min(img_h, y + h + pad_h)
    return x0, y0, max(2, x1 - x0), max(2, y1 - y0)

In [4]:
# Process VIA annotations and create crops
df = pd.read_csv(CSV_PATH, skiprows=SKIP_ROWS)
start_time = time.time()

# Group annotations by image for efficient processing
image_groups = defaultdict(list)
for _, row in df.iterrows():
    try:
        img_name = parse_file_list(row["file_list"])
        if not img_name: continue
        
        x, y, w, h = parse_box(row["spatial_coordinates"]) 
        behavior = extract_behavior(row["metadata"])
        image_groups[img_name].append((x, y, w, h, behavior))
    except: continue

# Process each image and generate crops
crops_written, missing_imgs = 0, 0
for img_name, annotations in tqdm(image_groups.items(), desc="Processing"):
    img_path = IMG_ROOT / img_name
    if not img_path.exists():
        missing_imgs += len(annotations)
        continue
        
    img = cv2.imread(str(img_path))
    if img is None:
        missing_imgs += len(annotations)
        continue
        
    img_h, img_w = img.shape[:2]
    
    for x, y, w, h, behavior in annotations:
        try:
            # Get padded crop coordinates
            cx, cy, cw, ch = get_padded_crop(x, y, w, h, img_w, img_h, PAD_FRACTION)
            crop = img[cy:cy+ch, cx:cx+cw]
            
            if crop.size == 0 or crop.shape[0] < 2 or crop.shape[1] < 2:
                continue
                
            # Save crop
            behavior_dir = OUT_ROOT / behavior
            behavior_dir.mkdir(parents=True, exist_ok=True)
            
            stem = Path(img_name).stem
            crop_name = f"{stem}_{cx}_{cy}_{cw}_{ch}.jpg"
            crop_path = behavior_dir / crop_name
            
            if cv2.imwrite(str(crop_path), crop):
                crops_written += 1
        except: continue

processing_time = time.time() - start_time

Processing: 100%|██████████| 3199/3199 [00:27<00:00, 118.43it/s]


In [5]:
# Results
behavior_counts = defaultdict(int)
for annotations in image_groups.values():
    for _, _, _, _, behavior in annotations:
        behavior_counts[behavior] += 1

print(f"Processed {len(image_groups)} images in {processing_time:.1f}s")
print(f"Created {crops_written} crops, {missing_imgs} missing")
print("Behavior distribution:")
for behavior, count in sorted(behavior_counts.items()):
    print(f"  {behavior}: {count}")
print(f"Output: {OUT_ROOT}")

Processed 3199 images in 27.6s
Created 25322 crops, 0 missing
Behavior distribution:
  drinking water: 744
  foraging: 5711
  lying down: 4518
  rumination: 6079
  stand: 8272
Output: workdir/crops_raw
